# Decision Tree — Model Development

## Project Overview
This notebook demonstrates an end-to-end implementation of a Decision Tree model to predict the target variable based on the given input features. The project covers data preprocessing, model training, evaluation, and model serialization for deployment.

## Business Problem
Many real-world business problems require predicting continuous values accurately, even when the relationship between input features and the target is non-linear. This project builds a Decision Tree model that learns these relationships and provides reliable predictions for unseen data.

## Workflow
1. Import required libraries
2. Load and inspect the dataset
3. Perform data preprocessing
4. Encode categorical features (if applicable)
5. Split the dataset into training and testing sets
6. Standardize the features
7. Train the Decision Tree model
8. Evaluate model performance using R² Score
9. Persist the trained model using Pickle

## Expected Outcome
A production-ready serialized model (`best_decision_tree_model.sav`) that can be deployed for making predictions on new data.

In [1]:
# Import library.
import pandas as pd

In [2]:
# Read the input dataset.
dataset = pd.read_csv('../data/50_Startups.csv')

In [3]:
# Load the first five rows of dataset.
dataset.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [4]:
# Convert the nominal data in the state column by one-hot encoding.
dataset = pd.get_dummies(dataset, drop_first=True).astype(int)

In [5]:
# Load the first five rows of dataset after one-hot encoding.
dataset.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_Florida,State_New York
0,165349,136897,471784,192261,0,1
1,162597,151377,443898,191792,0,0
2,153441,101145,407934,191050,1,0
3,144372,118671,383199,182901,0,1
4,142107,91391,366168,166187,1,0


In [6]:
# Assign columns to independent variable.
independent = dataset[['R&D Spend',	'Administration', 'Marketing Spend', 'State_Florida', 'State_New York']]

In [7]:
# Assign columns to dependent variable.
dependent = dataset[['Profit']]

In [8]:
# Split the training set and test set from the dataset (input).
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(independent, dependent, test_size=0.3, random_state=0)

In [9]:
# Create an instance of DecisionTreeRegressor.
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# Crete a decision tree model.
model = DecisionTreeRegressor(random_state=42)

# Define parameter values
param_grid = {
    "criterion": ["squared_error", "absolute_error", "poisson"],
    "splitter": ["best", "random"],
    "max_features": [2, 4, 0.5, 0.8, "sqrt", "log2", None]
}

# Create a GridSearchCV object to find the best combination of hyperparameters
grid = GridSearchCV(
    estimator=model,          # Machine Learning model to be tuned
    param_grid=param_grid,    # Dictionary containing hyperparameter values to test
    cv=5,                     # Perform 5-fold cross-validation
    scoring="r2",             # Use R² score to evaluate each parameter combination
    n_jobs=-1                 # Use all available CPU cores for faster execution
)

# Train the model using every hyperparameter combination and perform cross-validation
grid.fit(X_train, y_train)

# Display the hyperparameter combination that achieved the highest average R² score
print("Best Parameters:")
print(grid.best_params_)

# Display the best average cross-validation R² score
print("\nBest Score:")
print(grid.best_score_)

# Retrieve the model trained with the best hyperparameters
best_model = grid.best_estimator_

Best Parameters:
{'criterion': 'absolute_error', 'max_features': 4, 'splitter': 'best'}

Best Score:
0.8291916235807072


In [10]:
# Convert all GridSearchCV results into a Pandas DataFrame
results = pd.DataFrame(grid.cv_results_)

# Select only the columns required for analysis
results = results[
    [
        "param_criterion",      # Splitting criterion used
        "param_splitter",       # Splitting strategy used
        "param_max_features",   # Number of features considered at each split
        "mean_test_score",      # Average R² score across all cross-validation folds
        "rank_test_score"       # Ranking of each parameter combination (1 = Best)
    ]
]

# Rename the column names to make the output more readable
results.rename(columns={
    "param_criterion": "Criterion",
    "param_splitter": "Splitter",
    "param_max_features": "Max Features",
    "mean_test_score": "Mean R2 Score",
    "rank_test_score": "Rank"
}, inplace=True)

# Sort the results by rank so that the best-performing parameter combination appears first
results = results.sort_values(by="Rank", ascending=True)

# Save the GridSearchCV results to an Excel file
results.to_excel("../outputs/DT_GridSearch_Results.xlsx", index=False)

# Display the final sorted results
print(results)

         Criterion Splitter Max Features  Mean R2 Score  Rank
20  absolute_error     best          0.8       0.829192     1
16  absolute_error     best            4       0.829192     1
2    squared_error     best            4       0.771184     3
6    squared_error     best          0.8       0.771184     3
39         poisson   random         log2       0.730413     5
37         poisson   random         sqrt       0.730413     5
33         poisson   random          0.5       0.730413     5
29         poisson   random            2       0.730413     5
41         poisson   random         None       0.720368     9
12   squared_error     best         None       0.714209    10
13   squared_error   random         None       0.709523    11
26  absolute_error     best         None       0.691297    12
9    squared_error   random         sqrt       0.652248    13
1    squared_error   random            2       0.652248    13
11   squared_error   random         log2       0.652248    13
5    squ

In [11]:
# Save the model to pickle library.
import pickle

# Save the best trained model to a pickle file
with open("../models/final_tree_model.sav", "wb") as file:
    pickle.dump(best_model, file)

print("Model saved successfully!")

Model saved successfully!
